# Appendix A - Own extraction: PCAPNG to flow CSV (one CSV per capture)

The project previously used an own-extracted dataset from six MSCAD PCAPNG
captures. This notebook is the conversion step of that extraction and is kept for
traceability; results in Steps 1-12 of `MSCAD_Pipeline.ipynb` come from the
official MSCAD.csv.

**Purpose** — Convert the raw MSCAD packet captures into one labelled flow CSV
per capture, recovering the flow identifiers that the published Kaggle
`MSCAD.csv` does not contain.

**Input** — `C:\MSCAD2022\PCAPNG_ORIGINAL` (read-only). `C:\MSCAD2022\CSV_ORIGINAL`
holds the Kaggle file, used as a read-only column reference.

**Method** — `editcap` converts PCAPNG to PCAP; flows are then extracted with a
patched `cicflowmeter` and labelled from the documented MSCAD capture-to-attack
mapping (`config.CAPTURE_LABELS`) combined with the attacker IP addresses
measured from the traffic (any source sending at least `ATTACKER_THRESHOLD` times
the top sender's forward packets). Flows in an attack capture that do not involve
an attacker address are labelled `Benign`.

**Output** — `C:\MSCAD2022\CSVs\<capture>_flows.csv`, 73 columns: the 6 flow
identifiers (Src IP, Dst IP, Src Port, Dst Port, Protocol, Timestamp), the 66
CICFlowMeter features and `Label`. Conversion reports are written to
`Notebook\outputs\step1\`.

**Why** — The published feature table has no flow identifiers, so flows are
regenerated from the captures to recover them. Labels are never inferred from
packet contents, and the original captures are never modified.

The identifiers are retained for traceability and for the optional time-window
input mode in Step 8; they are excluded from model training, because the
attacker addresses determined the labels and training on them would leak
attacker identity.

---

## Install everything first (cmd, NOT PowerShell)

```
"C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m pip install scapy numpy scipy pandas requests tqdm jupyter pytest
"C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m pip install cicflowmeter==0.5.0 --no-deps --ignore-requires-python
```

`cicflowmeter 0.5.0` pins old dependencies and declares `python_requires < 3.13`,
so `--no-deps` and `--ignore-requires-python` are both required. Wireshark must
also be installed; it provides `editcap.exe`, `tshark.exe` and `capinfos.exe`.

## Start Jupyter and run the tests (cmd)

```
cd /d C:\Users\banot\MSCAD22-Multi-Stage-Cyber-Attack-Detection\Notebook
"C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m jupyter notebook
"C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m pytest tests -v
```

## Folder layout

Code and data are kept apart. The repository holds the code; `C:\MSCAD2022` holds
the data and every pipeline output.

```
C:\Users\banot\MSCAD22-Multi-Stage-Cyber-Attack-Detection\Notebook\
  config.py, src\, tests\, notebooks\      all code
  outputs\step1\                           conversion reports (this notebook)

C:\MSCAD2022\
  PCAPNG_ORIGINAL\   input .pcapng                      READ-ONLY
  PCAPS\             .pcap files produced by editcap
  CSVs\              <capture>_flows.csv  (+ raw\ before renaming)
  CSV_ORIGINAL\      the Kaggle MSCAD.csv               READ-ONLY, reference only
  outputs\step2 ... outputs\step12         the later pipeline steps
```

The output folder names keep the earlier numbering; the step-to-folder table at the
top of `MSCAD_Pipeline.ipynb` maps each one to its step.

```
```

Within this notebook the conversion is organised as stages A to D, which is the
naming used by `src\converter.py` and printed by the cells below. Every cell is
safe to re-run; all logic lives in `src\converter.py` and the cells only call it.


## 1. Setup

Point Python at the `Notebook` folder of the repository
(`C:\Users\banot\MSCAD22-Multi-Stage-Cyber-Attack-Detection\Notebook`), turn on
autoreload so that edits to `src\converter.py` take effect without restarting the
kernel, and import the module.


In [ ]:
import sys
from pathlib import Path

# The notebooks live in PROJECT_ROOT\notebooks, so when Jupyter is started in
# that folder the project root is one level up.  Starting Jupyter in the project
# root instead also works.  Nothing is hard-coded, so this is correct whether the
# code sits in C:\MSCAD2022\Notebook or in a git clone.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import config
from src import converter

print("project root   :", PROJECT_ROOT)
print("kernel python  :", sys.version.split()[0], "  <- the extractor runs HERE")
print("config         :", config.__file__)
print("converter      :", converter.__file__)

## 2. Config - every path, and whether it exists

`config.py` is the only file with hard-coded paths. If a row says `NO` for a
folder that should exist, fix it in `config.py` before going further.

In [ ]:
rep = converter.Report()
converter.stage_a_paths(rep)

print()
print("flow timeout       :", config.FLOW_TIMEOUT_S, "s")
print("idle timeout       :", config.IDLE_TIMEOUT_S, "s")
print("overwrite outputs  :", config.OVERWRITE)
print("progress every     :", f"{config.PROGRESS_EVERY:,}", "packets")
print("attacker threshold :", config.ATTACKER_THRESHOLD, "x the top sender")
print("output columns     :", len(config.FLOW_COLUMNS),
      f"({len(config.ID_COLUMNS)} identifiers + {len(config.KAGGLE_FEATURES)} features + Label)")

## 3. Tool check

Checks Python 3.13, the packages, `jupyter`, `pytest`, and `editcap` / `tshark` /
`capinfos` (`-v`). If something is missing it prints the exact cmd command to fix
it. `capinfos` is optional - it only makes packet counting faster.

In [ ]:
rep = converter.Report()
missing = converter.stage_a_environment(rep)
print()
print("config.PYTHON_EXE missing:", missing if missing else "nothing")

# The checks above probe config.PYTHON_EXE.  But the flow extractor runs in THIS
# kernel, which may be a different interpreter (e.g. a conda env).  scapy and
# pandas must therefore be installed in the kernel as well.
print()
print("=" * 78)
print("THE RUNNING KERNEL  (this is what Stage C/D actually uses)")
print("=" * 78)
print("  interpreter :", sys.executable)
print("  version     :", sys.version.split()[0])
kernel_missing = []
for name in ("scapy", "pandas", "numpy", "tqdm"):
    try:
        mod = __import__(name)
        print(f"  {name:<8} ok   {getattr(mod, '__version__', getattr(mod, 'VERSION', ''))}")
    except ImportError:
        kernel_missing.append(name)
        print(f"  {name:<8} MISSING")
if kernel_missing:
    print()
    print("  Install them INTO THIS KERNEL (cmd), then restart the kernel:")
    print(f'      "{sys.executable}" -m pip install ' + " ".join(kernel_missing))
else:
    print()
    print("  the kernel has everything Stage C/D needs")

## 4. Scan and inventory

Finds every `.pcapng` / `.pcap` under `PCAPNG_ORIGINAL` (recursive,
case-insensitive, so `.PCAPNG` matches too) and shows size, the label each stem
will get, and whether any two captures share a stem.

In [ ]:
rep = converter.Report()

capture_dir, why = converter.locate_capture_dir(rep)
print("capture folder:", capture_dir)
print("reason        :", why)
print()

captures = converter.find_captures(capture_dir) if capture_dir else []
converter.stage_a_inventory(rep, captures)

## 5. Kaggle reference and column diff

Reads `CSV_ORIGINAL\MSCAD.csv` (read-only) and prints its shape, its numbered
column names exactly as stored, and its Label counts. Then diffs those names
against `config.KAGGLE_FEATURES`.

**The Kaggle header always wins.** If anything differs, this cell prints a
ready-to-paste `KAGGLE_FEATURES = [...]` block for `config.py`.

In [ ]:
rep = converter.Report()

kaggle_df, kaggle_cols = converter.read_kaggle(rep)
rep.header("column names: config vs the Kaggle header")
kaggle_features = converter.compare_headers(rep, kaggle_cols)

## 6. PCAPNG to PCAP, with a packet-count check

`editcap -F pcap` per capture. A file that is already `.pcap` is **copied**, never
moved. Outputs that already exist are skipped unless `config.OVERWRITE = True`.

Each output is then verified: **the source and the converted file must have the
same packet count.** A mismatch is reported as `failed`, not as a warning - a
format conversion that loses packets is a broken conversion.

The table is saved to `outputs\step1\pcap_inventory.csv`.

In [ ]:
import pandas as pd

rep = converter.Report()
rows = converter.convert_all(captures, rep=rep)          # uses config.OVERWRITE

stage_b = pd.DataFrame(rows)
inventory_path = converter.write_inventory(captures)
print()
print("inventory saved to:", inventory_path)
stage_b

## 7. N-0 test run

The smallest capture first (214 packets), as a sanity check before the full run.
Shows packets read, flows, the shape, an **assert** that the header is exactly
`config.FLOW_COLUMNS`, `head()`, the Label counts and the runtime.

Writes `CSVs\raw\N-0_raw.csv` (exactly what the extractor produced, times still
in seconds) and `CSVs\N-0_flows.csv` (renamed, microseconds, labelled, reordered).

In [ ]:
# N-0 only.  Set OVERWRITE in config.py to True to redo an existing CSV.
import pandas as pd

n0 = next((c for c in captures if c.stem == "N-0"), None)
if n0 is None:
    print("N-0 not found - run cell 4 first.")
else:
    pcap_path = config.PCAP_DIR / converter.output_pcap_name(n0, captures)
    rep = converter.Report()
    df_n0, info_n0 = converter.process_capture(pcap_path, "N-0", rep=rep)

    # the header must be EXACTLY config.FLOW_COLUMNS, in that order
    missing = [c for c in config.FLOW_COLUMNS if c not in df_n0.columns]
    extra = [c for c in df_n0.columns if c not in config.FLOW_COLUMNS]
    assert list(df_n0.columns) == config.FLOW_COLUMNS, \
        f"header mismatch: missing {missing} / extra {extra}"

    converter.print_capture_summary(rep, df_n0, info_n0)
    print()
    print("header == config.FLOW_COLUMNS :", list(df_n0.columns) == config.FLOW_COLUMNS,
          f"({len(df_n0.columns)} columns)")
    print("runtime                       :", info_n0["runtime_s"], "s")
    print()
    print(df_n0[config.LABEL_COLUMN].value_counts())
    display(df_n0.head())

### 7b. N-0 key features vs the Kaggle Normal rows

Reference comparison only. The Kaggle file was produced by the Java
CICFlowMeter with its own settings, so the numbers are **not** expected to be
equal and nothing here is tuned to make them match.

In [ ]:
# describe() for a few key features, next to the Kaggle "Normal" rows.
# REFERENCE ONLY - nothing is tuned to make these match.
KEY = ["Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts",
       "Fwd Pkt Len Mean", "Flow IAT Mean", "Pkt Len Mean"]

print("OURS - N-0 flows")
display(df_n0[KEY].describe().T)

kaggle_path = config.find_kaggle_file()
if kaggle_path is None:
    print()
    print("No Kaggle file in", config.KAGGLE_DIR, "- so there is nothing to compare")
    print("against. That is not an error: Step 1 never needs it. Drop MSCAD.csv in")
    print("there and re-run this cell if you want the side-by-side.")
else:
    kag = (pd.read_excel(kaggle_path) if kaggle_path.suffix.lower() in (".xlsx", ".xls")
           else pd.read_csv(kaggle_path, low_memory=False))
    kag.columns = [str(c).strip() for c in kag.columns]
    normal = kag[kag[config.LABEL_COLUMN] == "Normal"]
    print()
    print(f"KAGGLE - Normal rows ({len(normal):,})")
    display(normal[[c for c in KEY if c in normal.columns]].describe().T)

## 8. Full conversion - all six captures

tqdm bar over the captures, plus a progress line every `config.PROGRESS_EVERY`
packets inside the big ones. Per capture it prints packets read, IPv4 packets,
the ARP/IPv6 gap, flows, dropped columns, the label method, the attacker IPs and
the **top 5 senders** so you can check the attacker detection by eye.

Writes `outputs\step1\conversion_log.csv` and `outputs\step1\label_report.csv`.
A capture that fails is logged with its exact error and reported at the end -
never skipped silently.

In [ ]:
# All six captures. ~4,500 packets/s was measured on synthetic traffic, so the
# ~1.01 M packets of the six captures should take roughly 4-6 minutes.
# A progress line prints every config.PROGRESS_EVERY packets.
rep = converter.Report()
log_rows, label_rows, frames = converter.run_all(captures, rep=rep)

import pandas as pd
display(pd.DataFrame(log_rows)[["stem", "status", "flows", "packets_total",
                                "packets_ipv4", "packets_non_ipv4", "runtime_s", "error"]])

## 9. Column validation

Asserts that **every** CSV in `C:\MSCAD2022\CSVs` has exactly
`config.FLOW_COLUMNS`, in exactly that order. Run it after cell 8. With no CSVs
yet it simply says so.

In [ ]:
import pandas as pd

csv_files = sorted(config.CSV_OUTPUT_DIR.glob("*_flows.csv"))
if not csv_files:
    print("no *_flows.csv in", config.CSV_OUTPUT_DIR, "- run cell 8 first (needs Stage C).")
else:
    all_ok = True
    for p in csv_files:
        header = list(pd.read_csv(p, nrows=0).columns)
        ok = header == config.FLOW_COLUMNS
        all_ok &= ok
        print(f"{p.name:<28} {len(header):>3} columns  {'OK' if ok else 'MISMATCH'}")
        if not ok:
            print("   missing:", [c for c in config.FLOW_COLUMNS if c not in header])
            print("   extra  :", [c for c in header if c not in config.FLOW_COLUMNS])
    assert all_ok, "at least one CSV does not match config.FLOW_COLUMNS"
    print()
    print(f"all {len(csv_files)} CSVs match config.FLOW_COLUMNS exactly")

## 10. Verification table

Per capture: packets in the file; IPv4 packets in the flows
(`Tot Fwd Pkts + Tot Bwd Pkts`) against an independent `tshark -Y ip` count -
these must match **exactly**, and the gap to the file total is ARP/IPv6; flows
written twice (same 5-tuple + start Timestamp, must be 0); NaN, negative,
min > max and invalid ports (all must be 0).

`inf` in `Flow Byts/s` / `Flow Pkts/s` for zero-duration flows is **normal** and
is counted, not fixed. Rows with identical 66-feature values are **expected** in
floods, scans and brute force, and are counted, not dropped.

Saved to `outputs\step1\verification.csv`.

In [ ]:
rep = converter.Report()
table = converter.verify_all(log_rows if "log_rows" in dir() else None, captures, rep=rep)

if len(table):
    display(table[["stem", "packets_in_pcap", "ipv4_packets_in_flows",
                   "tshark_ipv4_packets", "packet_counts_match", "non_ipv4_gap",
                   "flows", "duplicate_flow_keys", "nan_cells", "negative_cells",
                   "min_gt_max_rows", "invalid_ports", "inf_flow_byts_s",
                   "inf_flow_pkts_s", "identical_feature_rows",
                   "header_matches_flow_columns"]])

print()
print("=" * 78)
print("our label counts vs the Kaggle file (reference only)")
print("=" * 78)
converter.compare_with_kaggle(converter.Report())

## 11. Summary and logs

Lists what exists in `outputs\step1\`, `PCAPS\` and `CSVs\` right now.

In [ ]:
def summarise(folder, pattern="*"):
    files = sorted(p for p in folder.glob(pattern) if p.is_file())
    print(f"{folder}  ({len(files)} file(s))")
    for p in files:
        print(f"    {p.name:<34} {p.stat().st_size / 1024:>10.1f} KB")
    if not files:
        print("    (empty)")
    print()

summarise(config.OUTPUT_DIR)
summarise(config.PCAP_DIR, "*.pcap")
summarise(config.CSV_OUTPUT_DIR, "*_flows.csv")
summarise(config.RAW_DIR, "*_raw.csv")

## 12. Preview one CSV

Head, shape, dtypes and Label counts for the first output CSV.

In [ ]:
import pandas as pd

csv_files = sorted(config.CSV_OUTPUT_DIR.glob("*_flows.csv"))
if not csv_files:
    print("no *_flows.csv yet in", config.CSV_OUTPUT_DIR)
else:
    p = csv_files[0]
    df = pd.read_csv(p, low_memory=False)
    print(p.name, "| shape:", df.shape)
    print()
    print("dtypes:")
    print(df.dtypes.value_counts())
    print()
    print("Label counts:")
    print(df[config.LABEL_COLUMN].value_counts())
    display(df.head())